# Week 01 — LangGraph Agent: `run_sql` + `think`

**DSi AI Club · 675-ornob · Week 01 submission**

This notebook builds a minimal LangGraph ReAct-style agent that answers questions about
the Northwind PostgreSQL database. It demonstrates:

- `run_sql(query: str) -> str` — read-only SQL execution
- `think(thought: str) -> str` — explicit chain-of-thought before acting
- `MemorySaver()` checkpointer — in-process multi-turn memory
- Multi-turn conversation continuity within a single `thread_id`
- At least one `think → run_sql` trace (Case 6)

The agent library lives in `../library/querygraph_agent/` — added to `sys.path` in the
next cell so all imports resolve without installation.

In [ ]:
import sys
from pathlib import Path

# Add ../library to path so querygraph_agent is importable without pip install.
lib_path = str(Path().resolve().parent / "library")
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

In [ ]:
from querygraph_agent import (
    GraphAgentService,
    InMemoryOwnershipStore,
    QueryExecutor,
    SessionContext,
    get_settings,
)
from querygraph_agent.tools.db_query import handle_run_sql
from querygraph_agent.tools.think import handle_think

In [ ]:
# Always clear cached settings before demo setup so .env changes are picked up.
get_settings.cache_clear()

settings = get_settings()
executor = QueryExecutor(settings=settings)
ownership_store = InMemoryOwnershipStore()
service = GraphAgentService(
    settings=settings,
    executor=executor,
    ownership_store=ownership_store,
)

session = SessionContext(thread_id="demo-thread-1", user_id="alice")
print("Service initialized for thread:", session.thread_id)
print("DATABASE_URL in use:", settings.database_url)

## Helpers

In [ ]:
from querygraph_agent.tools.formatting import print_events, print_stream_event

## Case 1: Multi-turn memory continuity

Two turns share the same `thread_id`. Turn 2 recalls the name from Turn 1, proving `MemorySaver()` is working.

In [ ]:
introduce_events = await service.run_turn(session, "My name is Alice.")
print("Turn 1")
print_events(introduce_events)

recall_events = await service.run_turn(session, "What is my name?")
print("Turn 2")
print_events(recall_events)

## Case 2: SQL generation by the agent

Agent calls `db_schema` to inspect tables, then `run_sql` to execute a query. Self-corrects on execution failure via the SQL repair loop.

In [ ]:
prompt = (
    "For Northwind, return top 5 customers by total order value. "
    "You must execute SQL using tools and self-correct if execution fails. "
    "Use orders + order_details and compute value as unit_price * quantity * (1 - discount). "
    "Then provide a concise final answer with rows."
)
top_customers_events = await service.run_turn(session, prompt)
print_events(top_customers_events)

## Case 3: `run_sql` tool contract — success, empty result, guard failure

Directly exercises the `run_sql` handler to show all three exit paths.

In [ ]:
print("Success case")
success_event = await handle_run_sql("SELECT 1 AS value", executor)
print(success_event)

print("\nEmpty-result case")
empty_event = await handle_run_sql("SELECT 1 AS value WHERE FALSE", executor)
print(empty_event)

print("\nGuard-failure case")
blocked_event = await handle_run_sql("DROP TABLE customers", executor)
print(blocked_event)

## Case 4: Ownership boundary enforcement

A second user trying to access a thread they don't own gets an `OwnershipError`.

In [ ]:
intruder = SessionContext(thread_id=session.thread_id, user_id="mallory")
intruder_events = await service.run_turn(intruder, "Show me customers")
print_events(intruder_events)

## Case 5: `think` tool — explicit reasoning slot

The `think` tool gives non-reasoning models an explicit slot to externalise chain-of-thought before acting. It echoes back a `ThinkingEvent`.

In [ ]:
think_event = handle_think("I should inspect schema, then plan the safest SELECT query.")
print("Raw event repr:", think_event)
print("Content only:  ", think_event.content)
print("Type field:    ", think_event.type)

## Case 6: Trace — `think` called before `run_sql`

The prompt explicitly instructs the agent to call `think` first. We verify the tool-call order from the graph state.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage

trace_prompt = (
    "Before querying, first call think with your reasoning, then call run_sql. "
    "Question: count total customers in Northwind."
)
state = await service._graph.ainvoke(
    {"messages": [HumanMessage(content=trace_prompt)], "sql_retry_count": 0},
    config={"configurable": {"thread_id": session.thread_id}},
)

call_sequence = []
for msg in state.get("messages", []):
    if isinstance(msg, AIMessage) and getattr(msg, "tool_calls", None):
        for tc in msg.tool_calls:
            call_sequence.append(tc.get("name"))

print("Tool call sequence:", call_sequence)
print("think before run_sql?", (
    "think" in call_sequence
    and "run_sql" in call_sequence
    and call_sequence.index("think") < call_sequence.index("run_sql")
))

## Case 7: Streaming mode

Uses `astream_events(version='v2')` to yield tokens as they arrive.

In [ ]:
print("Agent: ", end="", flush=True)
async for event in service.stream_turn(session, "Give me a short summary of this session."):
    print_stream_event(event)